In [ ]:
# Install deps
!pip install -q pandas scikit-learn joblib tqdm datasets

In [ ]:
import os, zipfile, requests, sys
url = "https://github.com/vedangvatsa/ai-detection-at-scale/archive/refs/heads/main.zip"
r = requests.get(url)
open("repo.zip", "wb").write(r.content)
with zipfile.ZipFile("repo.zip") as z:
    z.extractall(".")
os.rename("ai-detection-at-scale-main", "ai-detection-at-scale")
os.chdir("ai-detection-at-scale")
sys.path.insert(0, '.')
print("Repo ready")

In [ ]:
# Download model assets
!python scripts/download_assets.py

In [ ]:
# MAGE Benchmark Evaluation
# MAGE = Machine-generated text detection with Adversarially Generated Examples
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score
from tool.feature_extractor import extract_features
import joblib

# Load stylometric model
model_data = joblib.load("models/detector_all.joblib")
model = model_data["model"] if isinstance(model_data, dict) else model_data

# Load MAGE from HuggingFace
from datasets import load_dataset
try:
    mage = load_dataset("yaful/MAGE")
    print(f"Columns: {mage['test'].column_names}")
    test_df = mage['test'].to_pandas()
except Exception as e:
    print(f"Could not load yaful/MAGE: {e}")
    print("Trying alternative...")
    mage = load_dataset("Hello-SimpleAI/HC3")
    print(f"Columns: {mage['train'].column_names}")
    # HC3 has human_answers and chatgpt_answers columns
    rows = []
    for item in mage['train']:
        for ans in item.get('human_answers', []):
            if isinstance(ans, str) and len(ans.strip()) > 20:
                rows.append({'text': ans, 'label': 0})
        for ans in item.get('chatgpt_answers', []):
            if isinstance(ans, str) and len(ans.strip()) > 20:
                rows.append({'text': ans, 'label': 1})
    test_df = pd.DataFrame(rows)

sample_size = min(5000, len(test_df))
test_sample = test_df.sample(n=sample_size, random_state=42)
print(f"Evaluating on {len(test_sample)} texts")

# Extract features and predict
features = []
labels = []
for i, row in test_sample.iterrows():
    text = row.get('text', row.get('article', ''))
    label_raw = row['label']
    # MAGE uses 0=machine, 1=human. We need 0=human, 1=AI.
    if isinstance(label_raw, str):
        label = 0 if label_raw.lower() in ('human', '1') else 1
    else:
        label = 1 - int(label_raw)  # Flip: 0->1 (AI), 1->0 (human)
    
    if not isinstance(text, str) or len(text.strip()) < 20:
        continue
    
    feats = extract_features(text, extended=False)
    if feats is None:
        continue
    
    X = np.array([feats[k] for k in ['mtld', 'sent_cv', 'self_mention_density', 'opener_ratio',
                                        'connector_density', 'hedge_density', 'mean_sent_len',
                                        'boost_density', 'char_entropy', 'rep_rate', 'punct_entropy']])
    features.append(X)
    labels.append(label)

features = np.array(features)
labels = np.array(labels)

# Predict
probs = model.predict_proba(features)[:, 1]
preds = (probs >= 0.5).astype(int)
auc = roc_auc_score(labels, probs)
acc = accuracy_score(labels, preds)
print(f"MAGE AUC: {auc:.4f}")
print(f"MAGE Accuracy: {acc:.4f}")

# Save results
results = pd.DataFrame([{'benchmark': 'MAGE', 'auc': auc, 'accuracy': acc, 'n_samples': len(labels)}])
results.to_csv('/kaggle/working/mage_results.csv', index=False)
print("Results saved")
